In [1]:
from simulator.sim3 import *
from exploration.env.func import Experiment, Env
from exploration.random.func import RANDOM
import numpy as np
from exploration.codegeneration import generate_instruction_sequence
from simulator.sim3 import print_contention_analysis
import pandas as pd

# Perform tests for the simulator

In [2]:
print("===================================================================")
print("1 core, 2 read cycles, same cache line, no dependency")
print("===================================================================")
# 1st RD => cache miss => DDR reads transaction 
# 2nd RD => cache hit => no DDR transaction
# Create instruction sequences
GlobalVar.clear_history()
inst0 = { 0: ('read', 0), 60: ('read', 17) }
inst1 = {}

exp=Experiment()
exp.load_instr(inst0, inst1)

results = exp.simulate(200, display_stats=True)

1 core, 2 read cycles, same cache line, no dependency

--- Simulation Stats ---
core0 {'level': 'L1', 'hits': 0, 'hits_read': 0, 'hits_write': 0, 'misses_read': 2, 'misses_write': 0, 'misses': 2, 'miss_rate': 1.0}
core1 {'level': 'L1', 'hits': 0, 'hits_read': 0, 'hits_write': 0, 'misses_read': 0, 'misses_write': 0, 'misses': 0, 'miss_rate': 0}
shared cache L2 {'level': 'L2', 'hits': 0, 'hits_read': 0, 'hits_write': 0, 'misses_read': 2, 'misses_write': 0, 'misses': 2, 'miss_rate': 1.0}
ddr hits [[[0. 0. 0. 0.]
  [0. 0. 0. 0.]]

 [[0. 0. 0. 0.]
  [0. 0. 0. 0.]]]
ddr miss [[[1. 0. 0. 0.]
  [0. 1. 0. 0.]]

 [[0. 0. 0. 0.]
  [0. 0. 0. 0.]]]


In [3]:
print("===================================================================")
print("1 core, 2 read cycles, same cache line, no dependency")
print("===================================================================")
# 1st RD => cache miss => DDR reads transaction 
# 2nd RD => cache hit => no DDR transaction
# Create instruction sequences
GlobalVar.clear_history()
inst0 = { 0: ('write', 0), 60: ('read', 17) }
inst1 = {}

exp=Experiment()
exp.load_instr(inst0, inst1)

results = exp.simulate(200, display_stats=True)

1 core, 2 read cycles, same cache line, no dependency

--- Simulation Stats ---
core0 {'level': 'L1', 'hits': 0, 'hits_read': 0, 'hits_write': 0, 'misses_read': 1, 'misses_write': 1, 'misses': 2, 'miss_rate': 1.0}
core1 {'level': 'L1', 'hits': 0, 'hits_read': 0, 'hits_write': 0, 'misses_read': 0, 'misses_write': 0, 'misses': 0, 'miss_rate': 0}
shared cache L2 {'level': 'L2', 'hits': 0, 'hits_read': 0, 'hits_write': 0, 'misses_read': 1, 'misses_write': 0, 'misses': 1, 'miss_rate': 1.0}
ddr hits [[[0. 0. 0. 0.]
  [0. 0. 0. 0.]]

 [[0. 0. 0. 0.]
  [0. 0. 0. 0.]]]
ddr miss [[[0. 0. 0. 0.]
  [0. 1. 0. 0.]]

 [[0. 0. 0. 0.]
  [0. 0. 0. 0.]]]


In [20]:
results

{'time_core0': 111,
 'time_core1': 0,
 'miss_ratios_detailled': array([[ 1., -0., -0., -0.],
        [-0.,  1., -0., -0.]]),
 'miss_ratios_global': array([ 1.,  1., -1., -1.]),
 'miss': array([[1., 0., 0., 0.],
        [0., 1., 0., 0.]]),
 'hits': array([[0., 0., 0., 0.],
        [0., 0., 0., 0.]]),
 'L2_miss': 2,
 'L2_hit': 0}

In [4]:
p0 = {1: ('write', 0),
 13: ('write', 10),
 17: ('write', 13),
 22: ('read', 11),
 24: ('read', 12),
 26: ('write', 19),
 28: ('write', 15),
 34: ('write', 8)}
#p0 = {
# 22: ('read', 11),
# 24: ('read', 12),
# 26: ('write', 19),
# 28: ('write', 15),
# 34: ('write', 8)}
p1 = {1: ('write', 7),
 9: ('write', 8),
 10: ('write', 19),
 25: ('write', 1),
 35: ('read', 3),
 50: ('read', 6)}

In [5]:
p0

{1: ('write', 0),
 13: ('write', 10),
 17: ('write', 13),
 22: ('read', 11),
 24: ('read', 12),
 26: ('write', 19),
 28: ('write', 15),
 34: ('write', 8)}

In [6]:
p0

{1: ('write', 0),
 13: ('write', 10),
 17: ('write', 13),
 22: ('read', 11),
 24: ('read', 12),
 26: ('write', 19),
 28: ('write', 15),
 34: ('write', 8)}

In [7]:
p1

{1: ('write', 7),
 9: ('write', 8),
 10: ('write', 19),
 25: ('write', 1),
 35: ('read', 3),
 50: ('read', 6)}

In [54]:
GlobalVar.clear_history()
exp=Experiment()
exp.load_instr(p0,  {})
results = exp.simulate(1000, display_stats=True)


--- Simulation Stats ---
core0 {'level': 'L1', 'hits': 4, 'misses': 4, 'miss_rate': 0.5}
core1 {'level': 'L1', 'hits': 0, 'misses': 0, 'miss_rate': 0}
shared cache L2 {'level': 'L2', 'hits': 0, 'misses': 0, 'miss_rate': 0}
ddr hits [[0. 0. 0. 0.]
 [0. 0. 0. 0.]]
ddr miss [[0. 0. 0. 0.]
 [0. 0. 0. 0.]]


In [55]:
print(pd.DataFrame(results['miss'],columns=[f'bank {j}' for j in range(4)],index = [f'row {j}' for j in range(2)]))

       bank 0  bank 1  bank 2  bank 3
row 0     0.0     0.0     0.0     0.0
row 1     0.0     0.0     0.0     0.0


In [56]:
print(pd.DataFrame(results['hits'],columns=[f'bank {j}' for j in range(4)],index = [f'row {j}' for j in range(2)]))

       bank 0  bank 1  bank 2  bank 3
row 0     0.0     0.0     0.0     0.0
row 1     0.0     0.0     0.0     0.0


In [9]:
exp.cache_stats_core_0['L2']['misses']

{'level': 'L2',
 'hits': 0,
 'misses': 2,
 'miss_rate': 1.0,
 'cache_miss_detailled': array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])}

In [3]:
GlobalVar.clear_history()
print("===================================================================")
print("1 core, 2 read cycles,same index, same tag, same bank, different rows, no dependency")
print("===================================================================")
# 1st RD => cache miss => DDR reads transaction 
# 2nd RD => cache miss => DDR reads transaction 
# Create instruction sequences
inst0 = { 0: ('read', 0), 60: ('read', 20) }
inst1 = {  }

exp=Experiment()
exp.load_instr(inst0, inst1)
results = exp.simulate(200,display_stats=True)
import pandas as pd
print('DDR miss ratio:')
pd.DataFrame(results['miss_ratios_detailled'],columns=[f'bank {j}' for j in range(4)],index = [f'row {j}' for j in range(2)])

1 core, 2 read cycles,same index, same tag, same bank, different rows, no dependency

--- Simulation Stats ---
core0 {'level': 'L1', 'hits': 0, 'misses': 2, 'miss_rate': 1.0}
core1 {'level': 'L1', 'hits': 0, 'misses': 0, 'miss_rate': 0}
shared cache L2 {'level': 'L2', 'hits': 0, 'misses': 2, 'miss_rate': 1.0}
ddr hits [[0. 0. 0. 0.]
 [0. 0. 0. 0.]]
ddr miss [[1. 0. 0. 0.]
 [1. 0. 0. 0.]]
DDR miss ratio:


,bank 0,bank 1,bank 2,bank 3
row 0,1.0,-0.0,-0.0,-0.0
row 1,1.0,-0.0,-0.0,-0.0


In [4]:
print("===================================================================")
print("1 core, 2 read cycles, same cache line, no dependency")
print("===================================================================")
# 1st RD => cache miss => DDR reads transaction 
# 2nd RD => cache hit => no DDR transaction
# Create instruction sequences
GlobalVar.clear_history()
inst0 = { 0: ('read', 0), 60: ('read', 0) }
inst1 = {}
#inst1 = { 0:('read', 4)}

exp=Experiment()
exp.load_instr(inst0, inst1)

results = exp.simulate(200, display_stats=True)
print('DDR miss ratio:')
pd.DataFrame(results['miss_ratios_detailled'],columns=[f'bank {j}' for j in range(4)],index = [f'row {j}' for j in range(2)])

1 core, 2 read cycles, same cache line, no dependency

--- Simulation Stats ---
core0 {'level': 'L1', 'hits': 0, 'misses': 2, 'miss_rate': 1.0}
core1 {'level': 'L1', 'hits': 0, 'misses': 0, 'miss_rate': 0}
shared cache L2 {'level': 'L2', 'hits': 0, 'misses': 2, 'miss_rate': 1.0}
ddr hits [[1. 0. 0. 0.]
 [0. 0. 0. 0.]]
ddr miss [[1. 0. 0. 0.]
 [0. 0. 0. 0.]]
DDR miss ratio:


,bank 0,bank 1,bank 2,bank 3
row 0,0.5,-0.0,-0.0,-0.0
row 1,-0.0,-0.0,-0.0,-0.0


In [12]:
GlobalVar.clear_history()
print("===================================================================")
print("1 core, 2 read cycles, different bank, no dependency")
print("===================================================================")
# bank nb = addr % num_banks, with num_banks=4
# row nb = addr // 16
# 1st RD in bank 0, row 0 
# 1st RD in bank 1, row 1
# 2nd RD => cache miss => DDR transaction
# Create instruction sequences

# 1st RD => cache miss => DDR reads transaction 
# 2nd RD => cache miss => DDR reads transaction 
inst0 = { 0: ('read', 0), 60: ('read', 17)}
inst1 = {  }

exp=Experiment()
exp.load_instr(inst0, inst1)

results = exp.simulate(100, display_stats=True)

1 core, 2 read cycles, different bank, no dependency

--- Simulation Stats ---
core0 {'level': 'L1', 'hits': 0, 'misses': 2, 'miss_rate': 1.0}
core1 {'level': 'L1', 'hits': 0, 'misses': 0, 'miss_rate': 0}
shared cache L2 {'level': 'L2', 'hits': 0, 'misses': 2, 'miss_rate': 1.0}
ddr hits [[0. 0. 0. 0.]
 [0. 0. 0. 0.]]
ddr miss [[1. 0. 0. 0.]
 [0. 1. 0. 0.]]


In [4]:
from exploration.load_file import load
import os

In [5]:
class test_bij:
    def __init__(self):
        self.num_banks = 4
    def _get_bank(self, addr):
        return addr % self.num_banks

    def _get_row(self, addr):
        return addr // 16 # Example: each row     covers 16 addresses (line_size is 4, so 4 cach    e lines per row for a 4-line_size cache)

In [6]:
folder = 'results'
path = os.path.join(folder,'imgep_run_1_10000')

In [47]:
content = load(path)
diff_time_core1 = content['memory_perf']['mutual']['diff_time_core1']
argmin_core1 = np.argmin(diff_time_core1)
print('core 1 mutual',content['memory_perf']['mutual']['time_core1'][argmin_core1])
print('core 1 iso',content['memory_perf']['core1']['time_core1'][argmin_core1])
print('core 0 mutual',content['memory_perf']['mutual']['time_core0'][argmin_core1])
print('core 0 iso',content['memory_perf']['core0']['time_core0'][argmin_core1])
tt = test_bij()
seq0 = content['memory_program']['core0'][argmin_core1]
seq1 = content['memory_program']['core1'][argmin_core1]     
seq0 = {k: seq0[k] for k in sorted(seq0.keys())}
seq1 = {k: seq1[k] for k in sorted(seq1.keys())}
print('seq core0', seq0)
print('seq core1', seq1)

results/imgep_run_1_10000_5.pkl
core 1 mutual 78.0
core 1 iso 162.0
core 0 mutual 108.0
core 0 iso 97.0
seq core0 {20: ('read', 17), 27: ('read', 15), 30: ('read', 16), 31: ('read', 2), 36: ('read', 9), 43: ('read', 15), 46: ('read', 8)}
seq core1 {6: ('read', 30), 8: ('read', 30), 11: ('read', 31), 18: ('read', 33), 27: ('read', 40), 36: ('read', 29), 40: ('read', 25), 42: ('write', 24), 45: ('write', 31), 47: ('read', 36)}


In [45]:
content['memory_perf'].keys()

dict_keys(['mutual', 'core0', 'core1'])

In [36]:
bank_row_seq = lambda seq:{cycle:{'bank':tt._get_bank(seq[cycle][1]),'row':tt._get_row(seq[cycle][1])} for cycle in seq}
r_b_seq0 = bank_row_seq(seq0)
r_b_seq1 = bank_row_seq(seq1)
print('core0',r_b_seq0)
print('core1',r_b_seq1)

core0 {20: {'bank': 1, 'row': 1}, 27: {'bank': 3, 'row': 0}, 30: {'bank': 0, 'row': 1}, 31: {'bank': 2, 'row': 0}, 36: {'bank': 1, 'row': 0}, 43: {'bank': 3, 'row': 0}, 46: {'bank': 0, 'row': 0}}
core1 {6: {'bank': 2, 'row': 1}, 8: {'bank': 2, 'row': 1}, 11: {'bank': 3, 'row': 1}, 18: {'bank': 1, 'row': 2}, 27: {'bank': 0, 'row': 2}, 36: {'bank': 1, 'row': 1}, 40: {'bank': 1, 'row': 1}, 42: {'bank': 0, 'row': 1}, 45: {'bank': 3, 'row': 1}, 47: {'bank': 0, 'row': 2}}


In [49]:
GlobalVar.clear_history()
exp=Experiment(num_addr=41)
exp.load_instr({}, seq1)
results = exp.simulate(400, display_stats=True)
print('time core1', results['time_core1'])


--- Simulation Stats ---
core0 {'level': 'L1', 'hits': 0, 'hits_read': 0, 'hits_write': 0, 'misses_read': 0, 'misses_write': 0, 'misses': 0, 'miss_rate': 0}
core1 {'level': 'L1', 'hits': 2, 'hits_read': 1, 'hits_write': 1, 'misses_read': 6, 'misses_write': 1, 'misses': 7, 'miss_rate': 0.7777777777777778}
shared cache L2 {'level': 'L2', 'hits': 0, 'hits_read': 0, 'hits_write': 0, 'misses_read': 6, 'misses_write': 1, 'misses': 7, 'miss_rate': 1.0}
ddr hits [[[0. 0. 0. 0.]
  [0. 0. 1. 0.]
  [0. 0. 0. 0.]]

 [[0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]]]
ddr miss [[[0. 0. 0. 0.]
  [0. 1. 1. 1.]
  [1. 1. 0. 0.]]

 [[0. 0. 0. 0.]
  [1. 0. 0. 0.]
  [0. 0. 0. 0.]]]
time core1 160


In [37]:
content = load(path)
diff_time_core1 = content['memory_perf']['mutual']['diff_time_core1']
argmax_core1 = np.argmax(diff_time_core1)
print('core 1 mutual',content['memory_perf']['mutual']['time_core1'][argmax_core1])
print('core 1 iso',content['memory_perf']['core1']['time_core1'][argmax_core1])
tt = test_bij()
seq0 = content['memory_program']['core0'][argmax_core1]
seq1 = content['memory_program']['core1'][argmax_core1]     
seq0 = {k: seq0[k] for k in sorted(seq0.keys())}
seq1 = {k: seq1[k] for k in sorted(seq1.keys())}
print('seq core0', seq0)
print('seq core1', seq1)

results/imgep_run_1_10000_5.pkl
core 1 mutual 168.0
core 1 iso 59.0
seq core0 {8: ('read', 6), 12: ('write', 4), 16: ('read', 18), 19: ('write', 13), 29: ('write', 19), 33: ('read', 8), 40: ('write', 16), 45: ('read', 12), 49: ('read', 12), 53: ('read', 12)}
seq core1 {2: ('read', 32), 4: ('write', 30), 7: ('read', 40), 8: ('read', 27), 16: ('read', 40), 30: ('read', 27), 39: ('read', 36), 44: ('read', 25), 52: ('read', 23), 55: ('write', 24)}


In [38]:
r_b_seq0 = bank_row_seq(seq0)
r_b_seq1 = bank_row_seq(seq1)

In [40]:
print(r_b_seq0)
print(r_b_seq1)

{8: {'bank': 2, 'row': 0}, 12: {'bank': 0, 'row': 0}, 16: {'bank': 2, 'row': 1}, 19: {'bank': 1, 'row': 0}, 29: {'bank': 3, 'row': 1}, 33: {'bank': 0, 'row': 0}, 40: {'bank': 0, 'row': 1}, 45: {'bank': 0, 'row': 0}, 49: {'bank': 0, 'row': 0}, 53: {'bank': 0, 'row': 0}}
{2: {'bank': 0, 'row': 2}, 4: {'bank': 2, 'row': 1}, 7: {'bank': 0, 'row': 2}, 8: {'bank': 3, 'row': 1}, 16: {'bank': 0, 'row': 2}, 30: {'bank': 3, 'row': 1}, 39: {'bank': 0, 'row': 2}, 44: {'bank': 1, 'row': 1}, 52: {'bank': 3, 'row': 1}, 55: {'bank': 0, 'row': 1}}


In [39]:

exp=Experiment(num_addr=41)
exp.load_instr(seq0, seq1)
results = exp.simulate(400, display_stats=True)
print('time core1', results['time_core1'])


--- Simulation Stats ---
core0 {'level': 'L1', 'hits': 4, 'hits_read': 3, 'hits_write': 1, 'misses_read': 3, 'misses_write': 3, 'misses': 6, 'miss_rate': 0.6}
core1 {'level': 'L1', 'hits': 2, 'hits_read': 1, 'hits_write': 1, 'misses_read': 7, 'misses_write': 1, 'misses': 8, 'miss_rate': 0.8}
shared cache L2 {'level': 'L2', 'hits': 2, 'hits_read': 0, 'hits_write': 2, 'misses_read': 10, 'misses_write': 0, 'misses': 10, 'miss_rate': 0.8333333333333334}
ddr hits [[[0. 0. 0. 0.]
  [0. 0. 0. 2.]
  [2. 0. 0. 0.]]

 [[0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]]]
ddr miss [[[1. 0. 1. 0.]
  [0. 0. 1. 1.]
  [2. 0. 0. 0.]]

 [[1. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]]]
time core1 170
